# Benchmarks — Hash Tables e Heaps (Python)
Este notebook contém **templates prontos** para medir desempenho de `dict`, `set` e `heapq`,
bem como de versões didáticas implementadas do zero. Execute célula a célula,
ajuste os tamanhos (`n`) e gere gráficos simples com `matplotlib`.

⚠️ *Dica:* rode múltiplas vezes e use amostragem para reduzir variação.


## Setup
Importe bibliotecas e helpers.

In [ ]:
import random, string, time, timeit, sys
import math
from collections import Counter
import matplotlib.pyplot as plt

def rand_str(k=12):
    return ''.join(random.choice(string.ascii_letters+string.digits) for _ in range(k))

def time_call(fn, repeats=5):
    vals = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        vals.append(time.perf_counter()-t0)
    vals.sort()
    return sum(vals)/len(vals), min(vals), max(vals)


## 1) dict vs HashTable (didática)
Abaixo uma implementação simples de hash table com encadeamento (para estudo), e comparação com `dict`.
Atenção: o `dict` do Python é altamente otimizado — a implementação didática serve para **aprender**, não para competir.

In [ ]:
from typing import Any, List, Tuple, Optional

class HashTable:
    def __init__(self, initial_capacity: int = 8, load_factor: float = 0.75):
        self._m = max(4, initial_capacity)
        self._buckets: List[List[Tuple[Any, Any]]] = [[] for _ in range(self._m)]
        self._n = 0
        self._load_factor = load_factor

    def _bucket_index(self, key: Any) -> int:
        return hash(key) % self._m

    def _need_resize(self) -> bool:
        return self._n / self._m > self._load_factor

    def _resize(self, new_m: int):
        old_items = [(k, v) for bucket in self._buckets for (k, v) in bucket]
        self._m = max(4, new_m)
        self._buckets = [[] for _ in range(self._m)]
        self._n = 0
        for k, v in old_items:
            self.insert(k, v)

    def insert(self, key: Any, value: Any) -> None:
        idx = self._bucket_index(key)
        bucket = self._buckets[idx]
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))
        self._n += 1
        if self._need_resize():
            self._resize(self._m * 2)

    def get(self, key: Any) -> Optional[Any]:
        idx = self._bucket_index(key)
        for k, v in self._buckets[idx]:
            if k == key:
                return v
        return None

    def delete(self, key: Any) -> bool:
        idx = self._bucket_index(key)
        bucket = self._buckets[idx]
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self._n -= 1
                return True
        return False

    def __len__(self):
        return self._n


In [ ]:
# Experimento básico: insert/get/delete em dict vs HashTable
def bench_dict_vs_hashtable(n=100_000):
    keys = [rand_str() for _ in range(n)]
    vals = list(range(n))

    # dict insert
    def dict_insert():
        d = {}
        for k, v in zip(keys, vals):
            d[k] = v
    t_dict_ins, *_ = time_call(dict_insert)

    # HashTable insert
    def ht_insert():
        ht = HashTable()
        for k, v in zip(keys, vals):
            ht.insert(k, v)
    t_ht_ins, *_ = time_call(ht_insert)

    return t_dict_ins, t_ht_ins

t_dict, t_ht = bench_dict_vs_hashtable(n=50_000)
print(f"Insert 50k — dict: {t_dict:.3f}s | HashTable(didática): {t_ht:.3f}s")

xs = ["dict", "HashTable"]
ys = [t_dict, t_ht]
plt.figure()
plt.bar(xs, ys)
plt.title("Tempo de inserção (50k)")
plt.ylabel("segundos")
plt.show()


## 2) Top-K: heap vs sort
Mantendo apenas `K` elementos com heap (min-heap de tamanho `K`) vs ordenar a coleção inteira.

In [ ]:
import heapq

def top_k_heap(seq, k):
    h = []
    for x in seq:
        if len(h) < k:
            heapq.heappush(h, x)
        else:
            if x > h[0]:
                heapq.heapreplace(h, x)
    return sorted(h, reverse=True)

def bench_topk(n=300_000, k=1000):
    data = [random.random() for _ in range(n)]
    def run_heap(): top_k_heap(data, k)
    def run_sort(): sorted(data)[-k:]
    t_heap, *_ = time_call(run_heap)
    t_sort, *_ = time_call(run_sort)
    return t_heap, t_sort

t_heap, t_sort = bench_topk(n=200_000, k=500)
print(f"Top-K (n=200k, k=500) — heap: {t_heap:.3f}s | sort: {t_sort:.3f}s")

plt.figure()
plt.bar(["heap"], [t_heap])
plt.bar(["sort"], [t_sort])
plt.title("Top-K: heap vs sort (tempo)")
plt.ylabel("segundos")
plt.show()


## 3) Priority Queue com lazy deletion
Template de fila de prioridade com `entry_finder` (dict) + remoção lógica.
Inclui medição de impacto de entradas inválidas e uma "compactação" periódica.

In [ ]:
import itertools, heapq
REMOVED = "<REMOVED>"

class PriorityQueue:
    def __init__(self):
        self.pq = []
        self.entry_finder = {}
        self.counter = itertools.count()

    def push(self, item, priority):
        if item in self.entry_finder:
            self.remove(item)
        entry = [priority, next(self.counter), item]
        self.entry_finder[item] = entry
        heapq.heappush(self.pq, entry)

    def remove(self, item):
        entry = self.entry_finder.pop(item, None)
        if entry: entry[-1] = REMOVED

    def pop_min(self):
        while self.pq:
            pr, _, it = heapq.heappop(self.pq)
            if it is not REMOVED:
                self.entry_finder.pop(it, None)
                return it, pr
        raise KeyError("empty")

    def valid_size(self):
        return len(self.entry_finder)

    def compact(self):
        valid = [e for e in self.pq if e[-1] is not REMOVED]
        self.pq = valid
        heapq.heapify(self.pq)

def bench_lazy(n=100_000, remove_ratio=0.4, update_ratio=0.3):
    pq = PriorityQueue()
    items = [f"t{i}" for i in range(n)]
    for it in items:
        pq.push(it, random.random())
    for it in items[:int(n*remove_ratio)]:
        pq.remove(it)
    for it in items[int(n*remove_ratio):int(n*(remove_ratio+update_ratio))]:
        pq.push(it, random.random())

    t0 = time.perf_counter()
    popped = 0
    try:
        while True:
            pq.pop_min()
            popped += 1
    except KeyError:
        pass
    t_total = time.perf_counter() - t0
    return t_total, popped

t, popped = bench_lazy(n=50_000)
print(f"Pop de todos os válidos (lazy): tempo={t:.3f}s, removidos fisicamente={popped}")


## 4) Heapify vs N inserções
Compara custo de `heapq.heapify(lista)` com empilhar elemento a elemento.

In [ ]:
def bench_heapify_vs_push(n=200_000):
    data = [random.random() for _ in range(n)]
    def via_heapify():
        h = data[:]
        heapq.heapify(h)
    def via_push():
        h = []
        for x in data:
            heapq.heappush(h, x)
    t_heapify, *_ = time_call(via_heapify)
    t_push, *_ = time_call(via_push)
    return t_heapify, t_push

t_h, t_p = bench_heapify_vs_push(n=150_000)
print(f"Heapify vs Push (n=150k): heapify={t_h:.3f}s | push={t_p:.3f}s")

plt.figure()
plt.bar(["heapify"], [t_h])
plt.bar(["push"], [t_p])
plt.title("Heapify vs N inserções")
plt.ylabel("segundos")
plt.show()


## Observações finais
- Ajuste `n`, `k` e número de repetições para o seu hardware.
- Cada gráfico está em um único plot, sem estilos de cor específicos (como exigido).
- Interprete medianas e variação; considere aquecimentos e GC.
